# 01 · Preprocess and split (leakage-free)
Builds per-subject windows from NinaPro DB2 with:
* **repetition-based split** (train reps 1,3,4,6 / test reps 2,5; windows straddling a boundary are dropped),
* **causal** Butterworth band-pass + rectification (what the firmware can actually compute),
* normalisation constants computed from **training repetitions only** (per-channel EMG scale, glove min/max) and saved for the firmware,
* targets: absolute angle at the last sample, previous-window angle, **per-stride delta**, window delta, restimulus, repetition, segment and time index (for ordered evaluation).

Outputs: `/kaggle/working/db2_processed/<subject>/data.npz` + `norm_stats.json`. Save this notebook's output as a dataset and attach it to the other notebooks.

In [ ]:
# --- locate emg_common.py: Kaggle utility script, attached dataset, or local folder ---
import os, sys, glob, json
for p in ['/kaggle/usr/lib/emg_common', '/kaggle/usr/lib/emg-common', '/kaggle/input/emg-common', '.', '..', '../kaggle_scripts', 'kaggle_scripts']:
    if os.path.exists(os.path.join(p, 'emg_common.py')):
        sys.path.insert(0, os.path.abspath(p)); break
else:
    raise ImportError('emg_common.py not found: add it as a Utility Script (File > Add utility script) or dataset')
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import emg_common as ec
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
WORK = os.environ.get('EMG_WORK', '/kaggle/working')           # EMG_* env vars exist only for local smoke tests
BASE = os.environ.get('EMG_BASE_PATH', ec.Config.base_path)
EPOCHS_OVERRIDE = int(os.environ['EMG_EPOCHS']) if 'EMG_EPOCHS' in os.environ else None
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| emg_common from', ec.__file__)

In [ ]:
cfg = ec.Config(base_path=BASE, processed_dir=f'{WORK}/db2_processed', n_subjects=int(os.environ.get('EMG_N_SUBJECTS', 10)),
                seq_len=200, stride=50, causal_filter=True, train_reps=(1, 3, 4, 6), test_reps=(2, 5))
subjects = ec.list_subjects(cfg)
print('subjects:', subjects)
print('window', cfg.seq_len, 'samples =', 1000 * cfg.seq_len / cfg.fs, 'ms | stride', cfg.stride, '=', 1000 * cfg.stride / cfg.fs, 'ms')

In [ ]:
npz_files = [ec.process_subject(cfg, s) for s in subjects]

In [ ]:
# ---- summary and sanity checks ----
rows = []
for f in npz_files:
    d = np.load(f)
    tr, te = d['split'] == 0, d['split'] == 1
    assert not (set(np.unique(d['rep'][tr])) & set(np.unique(d['rep'][te]))), 'train/test repetitions overlap'
    rows.append({'subject': os.path.basename(os.path.dirname(f)), 'windows': len(d['split']), 'train': int(tr.sum()), 'test': int(te.sum()),
                 'rest_frac_train': float((d['restim'][tr] == 0).mean()), 'train_reps': sorted(np.unique(d['rep'][tr]).tolist()),
                 'test_reps': sorted(np.unique(d['rep'][te]).tolist()), 'emg_mean': float(d['sEMG'].mean()), 'emg_max': float(d['sEMG'].max())})
summary = pd.DataFrame(rows); display(summary)
delta_scale = ec.delta_scale_from_train(npz_files); rest_pose = ec.rest_pose_from_train(npz_files); thr = ec.rest_energy_threshold(npz_files)
print('per-joint delta std (train):', np.round(delta_scale, 4))
print('rest pose (train):', np.round(rest_pose, 3), '| rest energy threshold:', round(thr, 4))
print('smallest delta std vs ap_fixed<10,3> LSB (1/128=0.0078):', round(float(delta_scale.min()), 4), '-> deltas MUST be standardised before quantisation')
json.dump({'summary': rows, 'delta_scale': delta_scale.tolist(), 'rest_pose': rest_pose.tolist(), 'rest_energy_threshold': thr, 'cfg': cfg.to_dict()},
          open(f'{cfg.processed_dir}/dataset_summary.json', 'w'), indent=2)

In [ ]:
# ---- visual check: one window and the ordered glove trace of a test run ----
d = np.load(npz_files[0]); runs = ec.ordered_runs(npz_files[0], 'test'); r = max(runs, key=lambda x: len(x['angles']))
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
ax[0].plot(d['sEMG'][100]); ax[0].set_title('one window: mu-law rectified EMG (12 ch)')
ax[1].plot(r['angles'][:400]); ax[1].set_title(f'ordered test run: normalised angles ({len(r["angles"])} windows)')
ax[2].plot(r['delta_stride'][:400]); ax[2].set_title('per-stride deltas (integration targets)')
plt.tight_layout(); plt.savefig(f'{cfg.processed_dir}/preview.png', dpi=120); plt.show()

### Notes for the firmware
`norm_stats.json` per subject holds `emg_scale` (per channel), `glove_min`, `glove_max`, `mu`. The export notebook additionally writes the causal SOS filter
coefficients and `DELTA_SCALE` to `normalization_constants.h`. Nothing here uses future samples, so the PS can reproduce the pipeline sample by sample.